## Helsinki opus-mt baseline — English ↔ Arabic, with BLEU, chrF++, and COMET

Mirrors the NLLB zero-shot pipeline so scores are directly comparable. On Kaggle: enable the **GPU** accelerator and **internet** (the opus-mt checkpoints + COMET download from the hub).

**Note on "zero-shot":** unlike NLLB (one multilingual model), Helsinki `opus-mt` ships *one bilingual checkpoint per direction* trained directly on en→ar / ar→en. So this is a **direct supervised baseline**, not zero-shot in the strict sense — the pipeline is identical, only the label differs.

## 1. Install & Imports

In [1]:
# Marian/opus-mt needs sentencepiece (+ sacremoses for some checkpoints).
!pip install -q sentencepiece sacremoses sacrebleu

In [2]:
import subprocess, sys

def pip(args):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + args.split(),
                   check=True)

# Pin a known-good COMET + its dependency stack
pip("unbabel-comet==2.2.2")
pip("datasets==2.19.0")
pip("fsspec==2024.3.1")

print("Done — now RESTART THE KERNEL before continuing.")

Done — now RESTART THE KERNEL before continuing.


In [3]:
import gc
import torch
import pandas as pd
import sacrebleu
from transformers import MarianMTModel, MarianTokenizer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

2026-06-09 02:09:38.424291: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1780970978.655543     650 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780970978.722319     650 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1780970979.265228     650 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780970979.265260     650 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780970979.265263     650 computation_placer.cc:177] computation placer alr

Device: cuda
GPU: Tesla T4


# 2. Config

In [10]:
# ============================================================
# CONFIG
# ============================================================
class Config:
    # --- Models: opus-mt is bilingual, so ONE checkpoint per direction ---
    EN2AR_MODEL = "Helsinki-NLP/opus-mt-tc-big-en-ar"   # English -> Arabic
    AR2EN_MODEL = "Helsinki-NLP/opus-mt-tc-big-ar-en"   # Arabic  -> English

    # --- COMET model (reference-based, multilingual: supports en & ar) ---
    COMET_MODEL = "Unbabel/wmt22-comet-da"

    # --- Data: single CSV with "en" and "ar" columns ---
    DATA_CSV   = "/kaggle/input/datasets/mishbhaul/english-arabic-devtest/devtest.csv"  # <-- point to your uploaded dataset
    EN_COL     = "en"
    AR_COL     = "ar"
    SPLIT_COL  = "split"      # optional; set to None to skip filtering
    SPLIT_VALUE = "devtest"   # keep only this split if SPLIT_COL exists

    # --- Generation ---
    BATCH_SIZE   = 16     # lower to 8 if you hit OOM
    NUM_BEAMS    = 4
    MAX_LENGTH   = 256
    COMET_BATCH  = 16     # COMET runs its own batching

    # --- Precision ---
    # opus-mt (Marian) can produce garbage under fp16 beam search on some
    # checkpoints; fp32 is safe and these models are tiny (~300M). Flip to
    # True only if you've verified output quality.
    USE_FP16 = False

    # --- Output ---
    OUTPUT_DIR = "/kaggle/working"


CFG = Config()

# 3. Data Loading

In [5]:
# ============================================================
# DATA LOADING
# ============================================================
def load_parallel_csv(path, en_col="en", ar_col="ar",
                      split_col=None, split_value=None):
    """Load aligned English/Arabic sentences from a single CSV.
    Returns (english_list, arabic_list)."""
    df = pd.read_csv(path, encoding="utf-8")

    missing = {en_col, ar_col} - set(df.columns)
    assert not missing, (
        f"CSV missing columns: {missing}. Found: {list(df.columns)}"
    )

    # Optional split filter (the devtest.csv carries a 'split' column)
    if split_col and split_col in df.columns and split_value is not None:
        df = df[df[split_col] == split_value]

    n_before = len(df)
    df = df.dropna(subset=[en_col, ar_col])
    df[en_col] = df[en_col].astype(str).str.strip()
    df[ar_col] = df[ar_col].astype(str).str.strip()
    df = df[(df[en_col] != "") & (df[ar_col] != "")]
    n_after = len(df)

    print(f"Loaded {n_after} valid pairs from {path} "
          f"(dropped {n_before - n_after})")
    return df[en_col].tolist(), df[ar_col].tolist()

# 4. Model Loading

In [11]:
# ============================================================
# MODEL LOADING
# ============================================================
def load_helsinki(model_name):
    """Load a Helsinki opus-mt (Marian) tokenizer + model for ONE direction.
    Direction is baked into the checkpoint — no language codes needed."""
    print(f"Loading {model_name} ...")
    tokenizer = MarianTokenizer.from_pretrained(model_name)
    dtype = torch.float16 if (CFG.USE_FP16 and DEVICE == "cuda") else torch.float32
    model = MarianMTModel.from_pretrained(model_name, dtype=dtype).to(DEVICE)
    model.eval()
    print("  loaded.")
    return tokenizer, model


def load_comet(comet_model):
    """Download + load the COMET evaluation model."""
    from comet import download_model, load_from_checkpoint
    print(f"Loading COMET: {comet_model} ...")
    ckpt = download_model(comet_model)
    model = load_from_checkpoint(ckpt)
    print("COMET loaded.")
    return model


def free_model(model):
    """Release a translation model from GPU memory between directions.
    We load two opus-mt checkpoints + COMET on one T4, so freeing the
    finished direction keeps headroom."""
    del model
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

# 5. Translation Function

In [7]:
# ============================================================
# TRANSLATION  (Helsinki / MarianMT)
# ============================================================
@torch.no_grad()
def translate(sentences, tokenizer, model, cfg):
    """Batch-translate with a Helsinki opus-mt model.
    Direction is fixed by the loaded checkpoint, so there are no
    language codes or forced_bos_token_id to set."""
    outputs = []
    for i in range(0, len(sentences), cfg.BATCH_SIZE):
        batch = sentences[i : i + cfg.BATCH_SIZE]
        enc = tokenizer(
            batch, return_tensors="pt", padding=True,
            truncation=True, max_length=cfg.MAX_LENGTH,
        ).to(DEVICE)

        gen = model.generate(
            **enc,
            num_beams=cfg.NUM_BEAMS,
            max_length=cfg.MAX_LENGTH,
            early_stopping=True,
        )
        outputs.extend(
            tokenizer.batch_decode(gen, skip_special_tokens=True)
        )
        done = min(i + cfg.BATCH_SIZE, len(sentences))
        print(f"  {done}/{len(sentences)}", end="\r")

    print()
    return outputs

# 6. Evaluation Metrics

In [8]:
# ============================================================
# EVALUATION
# ============================================================
def eval_surface(hypotheses, references):
    """sacreBLEU BLEU + chrF++.
    chrF++ = chrF with word n-grams (word_order=2). It's the
    standard WMT variant and more reliable than BLEU for
    morphologically rich targets like Arabic."""
    bleu   = sacrebleu.corpus_bleu(hypotheses, [references])
    chrfpp = sacrebleu.corpus_chrf(hypotheses, [references],
                                   word_order=2)   # the "++"
    return bleu.score, chrfpp.score


def eval_comet(sources, hypotheses, references, comet_model, cfg):
    """Reference-based COMET score (0-1, higher better)."""
    data = [
        {"src": s, "mt": h, "ref": r}
        for s, h, r in zip(sources, hypotheses, references)
    ]
    out = comet_model.predict(
        data,
        batch_size=cfg.COMET_BATCH,
        gpus=1 if DEVICE == "cuda" else 0,
    )
    return out["system_score"], out["scores"]   # system avg, per-sentence


def evaluate(sources, hypotheses, references, direction_name,
             comet_model, cfg):
    """Run all three metrics for one direction."""
    bleu, chrfpp = eval_surface(hypotheses, references)
    comet_sys, comet_each = eval_comet(
        sources, hypotheses, references, comet_model, cfg
    )

    print(f"\n=== {direction_name} ===")
    print(f"BLEU   : {bleu:.2f}")
    print(f"chrF++ : {chrfpp:.2f}")
    print(f"COMET  : {comet_sys:.4f}")

    return {
        "direction": direction_name,
        "bleu": round(bleu, 2),
        "chrf++": round(chrfpp, 2),
        "comet": round(comet_sys, 4),
    }, comet_each


def save_outputs(src, hyp, ref, comet_each, direction_tag, cfg):
    """Per-sentence dump with COMET score for error inspection."""
    df = pd.DataFrame({
        "source": src,
        "hypothesis": hyp,
        "reference": ref,
        "comet": [round(c, 4) for c in comet_each],
    })
    path = f"{cfg.OUTPUT_DIR}/helsinki_{direction_tag}.csv"
    df.to_csv(path, index=False, encoding="utf-8-sig")
    print(f"Saved outputs -> {path}")
    return df

# 7. Main Function

In [12]:
# ============================================================
# MAIN
# ============================================================
def run():
    # Load data once (en and ar are aligned)
    english, arabic = load_parallel_csv(
        CFG.DATA_CSV, CFG.EN_COL, CFG.AR_COL,
        CFG.SPLIT_COL, CFG.SPLIT_VALUE,
    )

    # Load COMET once (shared across both directions)
    comet = load_comet(CFG.COMET_MODEL)

    results = []

    # ---- English -> Arabic ----
    tok, mdl = load_helsinki(CFG.EN2AR_MODEL)
    hyp_ar = translate(english, tok, mdl, CFG)
    free_model(mdl)                       # free before COMET runs on GPU
    row, comet_each = evaluate(english, hyp_ar, arabic,
                               "English -> Arabic", comet, CFG)
    results.append(row)
    save_outputs(english, hyp_ar, arabic, comet_each, "en2ar", CFG)

    # ---- Arabic -> English ----
    tok, mdl = load_helsinki(CFG.AR2EN_MODEL)
    hyp_en = translate(arabic, tok, mdl, CFG)
    free_model(mdl)
    row, comet_each = evaluate(arabic, hyp_en, english,
                               "Arabic -> English", comet, CFG)
    results.append(row)
    save_outputs(arabic, hyp_en, english, comet_each, "ar2en", CFG)

    # ---- Summary ----
    summary = pd.DataFrame(results)
    summary.to_csv(f"{CFG.OUTPUT_DIR}/helsinki_summary.csv", index=False)
    print("\n" + "=" * 45)
    print(summary.to_string(index=False))
    return summary


summary = run()

Loaded 500 valid pairs from /kaggle/input/datasets/mishbhaul/english-arabic-devtest/devtest.csv (dropped 0)
Loading COMET: Unbabel/wmt22-comet-da ...


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.migration.utils:Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.6.4. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../../root/.cache/huggingface/hub/models--Unbabel--wmt22-comet-da/snapshots/2760a223ac957f30acfb18c8aa649b01cf1d75f2/checkpoints/model.ckpt`


COMET loaded.
Loading Helsinki-NLP/opus-mt-tc-big-en-ar ...


/usr/local/lib/python3.12/dist-packages/pytorch_lightning/core/saving.py:197: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']


...

model.safetensors:   0%|          | 0.00/603M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

  loaded.
  500/500


INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
Predicting DataLoader 0: 100


=== Arabic -> English ===
BLEU   : 22.91
chrF++ : 48.02
COMET  : 0.8248
Saved outputs -> /kaggle/working/helsinki_ar2en.csv

        direction  bleu  chrf++  comet
English -> Arabic 15.74   47.90 0.8660
Arabic -> English 22.91   48.02 0.8248
